# Baseline и эксперименты V1 / V2 / V3

Здесь воспроизводится экспериментальная часть проекта. Метрики ниже зафиксированы по уже выполненным запускам

## 1. Общая настройка

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from collections import Counter
from datasets import load_dataset, ClassLabel
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from src.project_utils import (
    SEED, MODEL_NAME, DATASET_NAME, NUM_LABELS, MAX_LENGTH,
    set_seed, clean_html, build_text, build_text_with_attributes,
    compute_metrics, make_model, WeightedTrainer, predict_category,
)

set_seed(SEED)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())


c:\Users\alive\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.14.0+cu126
CUDA: True


In [2]:
dataset = load_dataset(DATASET_NAME)
print(dataset)
print(dataset["train"].column_names)
print("Rows:", dataset["train"].num_rows)

c:\Users\alive\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\alive\.cache\huggingface\hub\datasets--nyuuzyou--ke-products. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 2059468/2059468 [00:25<00:00, 82082.75 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'category', 'parent_category', 'grandparent_category', 'greatgrandparent_category', 'rating', 'description', 'comments', 'attributes', 'seller', 'seller_description', 'adultCategory'],
        num_rows: 2059468
    })
})
['id', 'title', 'category', 'parent_category', 'grandparent_category', 'greatgrandparent_category', 'rating', 'description', 'comments', 'attributes', 'seller', 'seller_description', 'adultCategory']
Rows: 2059468


In [3]:
target_column = "greatgrandparent_category"
none_count = sum(value is None for value in dataset["train"][target_column])
print("Total:", dataset["train"].num_rows)
print("Missing target:", none_count)
print("Valid target:", dataset["train"].num_rows - none_count)

Total: 2059468
Missing target: 657606
Valid target: 1401862


In [4]:
dataset = dataset.filter(lambda example: example[target_column] is not None)
label_names = sorted(set(dataset["train"][target_column]))
label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for label, idx in label2id.items()}
print("Classes:", len(label_names))

Filter: 100%|██████████| 2059468/2059468 [00:19<00:00, 107981.99 examples/s]


Classes: 39


In [5]:
dataset = dataset.map(build_text)
dataset = dataset.map(lambda example: {"labels": label2id[example[target_column]]})
dataset = dataset.cast_column("labels", ClassLabel(num_classes=len(label_names), names=label_names))
print(dataset["train"][0]["text"])

Casting the dataset: 100%|██████████| 1401862/1401862 [00:03<00:00, 387107.34 examples/s]


Свитер вязанный осень-зима 2017 Очень хороший свитер


In [6]:
train_test = dataset["train"].train_test_split(test_size=0.10, seed=SEED, stratify_by_column="labels")
val_test = train_test["test"].train_test_split(test_size=0.5, seed=SEED, stratify_by_column="labels")
train_dataset = train_test["train"]
val_dataset = val_test["train"]
test_dataset = val_test["test"]
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 1261675
Validation: 70093
Test: 70094


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_model_dataset = train_dataset.map(tokenize, batched=True)
val_model_dataset = val_dataset.map(tokenize, batched=True)
test_model_dataset = test_dataset.map(tokenize, batched=True)
model_columns = ["input_ids", "attention_mask", "labels"]

def keep_model_columns(split):
    split = split.remove_columns([c for c in split.column_names if c not in model_columns])
    split.set_format("torch", columns=model_columns)
    return split

train_model_dataset = keep_model_columns(train_model_dataset)
val_model_dataset = keep_model_columns(val_model_dataset)
test_model_dataset = keep_model_columns(test_model_dataset)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(train_model_dataset.column_names)

Map: 100%|██████████| 70094/70094 [00:15<00:00, 4500.79 examples/s]

['labels', 'input_ids', 'attention_mask']


In [8]:
sample = train_model_dataset.select(range(5000))
lengths = [len(ids) for ids in sample["input_ids"]]
print("Min:", min(lengths))
print("Mean:", sum(lengths)/len(lengths))
print("Max:", max(lengths))
print("At max length:", sum(length >= MAX_LENGTH for length in lengths))
print("Share at max length:", sum(length >= MAX_LENGTH for length in lengths)/len(lengths))

Min: 4
Mean: 114.4854
Max: 256
At max length: 516
Share at max length: 0.1032


## 6. Общие метрики

In [9]:
def print_metrics(name, metrics):
    print(name)
    for key, value in metrics.items():
        print(f"{key}: {value:.6f}")

## 7. V1 — baseline

Вход: `title + description`. Loss: обычная Cross-Entropy

In [10]:
V1_DIR = Path("../models/v1_baseline")
RETRAIN_MODELS = False

model_v1 = make_model()
training_args_v1 = TrainingArguments(
    output_dir=str(V1_DIR), num_train_epochs=5, learning_rate=2e-5,
    per_device_train_batch_size=64, per_device_eval_batch_size=128,
    weight_decay=0.01, eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1_macro",
    greater_is_better=True, fp16=torch.cuda.is_available(), logging_steps=100,
    save_total_limit=2, seed=SEED, data_seed=SEED, report_to="none",
)
trainer_v1 = Trainer(
    model=model_v1, args=training_args_v1, train_dataset=train_model_dataset,
    eval_dataset=val_model_dataset, processing_class=tokenizer,
    data_collator=data_collator, compute_metrics=compute_metrics,
)
if RETRAIN_MODELS:
    trainer_v1.train()
    trainer_v1.save_model(V1_DIR)
    tokenizer.save_pretrained(V1_DIR)

c:\Users\alive\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\alive\.cache\huggingface\hub\models--cointegrated--rubert-tiny2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 55/55 [00:00<00:00, 11016.03it/s]
[transformers] BertF

In [11]:
v1_validation = {"accuracy": 0.968242, "f1_macro": 0.938793, "f1_weighted": 0.968488}
print_metrics("V1 validation", v1_validation)

V1 validation
accuracy: 0.968242
f1_macro: 0.938793
f1_weighted: 0.968488


## 8. V2 — weighted loss

Из-за дисбаланса классов используем сглаженные веса классов. Выбор модели выполняется по validation Macro F1

In [12]:
labels = np.array(train_model_dataset["labels"])
class_counts = np.bincount(labels, minlength=NUM_LABELS)
raw_weights = len(labels) / (NUM_LABELS * class_counts)
class_weights = torch.tensor(raw_weights, dtype=torch.float32)
smooth_weights = torch.sqrt(class_weights)
smooth_weights = smooth_weights / smooth_weights.mean()
smooth_weights = torch.clamp(smooth_weights, min=0.5, max=5.0)

weights_table = pd.DataFrame({"class": label_names, "count": class_counts, "weight": smooth_weights.numpy()})
weights_table.sort_values("count").head(10)

,class,count,weight
18,Наушники и аудиотехника,185,4.420615
10,Детский спорт и активный отдых,338,3.270470
6,Гигиена и подгузники,606,2.442487
22,Офисная техника,680,2.305760
30,Техника для кухни,683,2.300690
16,Компьютерная техника,875,2.032659
37,Электроинструменты,1369,1.625050
28,Текстиль,1391,1.612148
11,Женская одежда,1694,1.460871
4,Бытовая химия и личная гигиена,1695,1.460440


In [13]:
V2_DIR = Path("../models/v2_weighted")
model_v2 = make_model()
training_args_v2 = TrainingArguments(
    output_dir=str(V2_DIR), num_train_epochs=5, learning_rate=2e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=16,
    weight_decay=0.01, lr_scheduler_type="linear", warmup_ratio=0.1,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1_macro", greater_is_better=True, logging_steps=100,
    seed=SEED, data_seed=SEED, report_to="none",
)
trainer_v2 = WeightedTrainer(
    model=model_v2, args=training_args_v2, train_dataset=train_model_dataset,
    eval_dataset=val_model_dataset, processing_class=tokenizer,
    data_collator=data_collator, compute_metrics=compute_metrics, class_weights=smooth_weights,
)
if RETRAIN_MODELS:
    trainer_v2.train()
    trainer_v2.save_model(V2_DIR)
    tokenizer.save_pretrained(V2_DIR)

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 10351.19it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the c

In [14]:
v2_validation = {"accuracy": 0.972408, "f1_macro": 0.952999, "f1_weighted": 0.972413}
print_metrics("V2 validation", v2_validation)

V2 validation
accuracy: 0.972408
f1_macro: 0.952999
f1_weighted: 0.972413


## 9. V3 — attributes ablation

Проверяем добавление `attributes` при тех же данных и weighted loss. Эксперимент проведён на полном train

In [15]:
train_raw_v3 = train_dataset.map(build_text_with_attributes, desc="Building train V3")
val_raw_v3 = val_dataset.map(build_text_with_attributes, desc="Building validation V3")

def tokenize_v3(batch):
    return tokenizer(batch["text_v3"], truncation=True, max_length=MAX_LENGTH)

train_model_v3 = train_raw_v3.map(tokenize_v3, batched=True)
val_model_v3 = val_raw_v3.map(tokenize_v3, batched=True)
train_model_v3 = keep_model_columns(train_model_v3)
val_model_v3 = keep_model_columns(val_model_v3)

Map: 100%|██████████| 70093/70093 [00:09<00:00, 7252.03 examples/s]


In [16]:
V3_DIR = Path("../models/v3_attributes")
model_v3 = make_model()
training_args_v3 = TrainingArguments(
    output_dir=str(V3_DIR), num_train_epochs=5, learning_rate=2e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=16,
    weight_decay=0.01, lr_scheduler_type="linear", warmup_ratio=0.1,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1_macro", greater_is_better=True, logging_steps=100,
    seed=SEED, data_seed=SEED, report_to="none",
)
trainer_v3 = WeightedTrainer(
    model=model_v3, args=training_args_v3, train_dataset=train_model_v3,
    eval_dataset=val_model_v3, processing_class=tokenizer,
    data_collator=data_collator, compute_metrics=compute_metrics, class_weights=smooth_weights,
)
if RETRAIN_MODELS:
    trainer_v3.train()
    trainer_v3.save_model(V3_DIR)
    tokenizer.save_pretrained(V3_DIR)

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 7454.97it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the ch

In [17]:
v3_validation = {"accuracy": 0.972594, "f1_macro": 0.952233, "f1_weighted": 0.972604}
print_metrics("V3 validation", v3_validation)

V3 validation
accuracy: 0.972594
f1_macro: 0.952233
f1_weighted: 0.972604


## 10. Сравнение моделей

In [18]:
comparison = pd.DataFrame([
    {"Model":"V1","Input":"title + description","Loss":"Cross-Entropy","Accuracy":0.968242,"Macro F1":0.938793,"Weighted F1":0.968488},
    {"Model":"V2","Input":"title + description","Loss":"Weighted CE","Accuracy":0.972408,"Macro F1":0.952999,"Weighted F1":0.972413},
    {"Model":"V3","Input":"title + description + attributes","Loss":"Weighted CE","Accuracy":0.972594,"Macro F1":0.952233,"Weighted F1":0.972604},
])
comparison

,Model,Input,Loss,Accuracy,Macro F1,Weighted F1
0,V1,title + description,Cross-Entropy,0.968242,0.938793,0.968488
1,V2,title + description,Weighted CE,0.972408,0.952999,0.972413
2,V3,title + description + attributes,Weighted CE,0.972594,0.952233,0.972604


**Выбрана V2** по максимальному validation Macro F1: `0.9530`. V3 практически совпадает с V2 по Accuracy и Weighted F1, но имеет немного меньший Macro F1; поэтому attributes не дают значимого улучшения в данном эксперименте

## Вывод

Weighted loss заметно улучшил Macro F1 относительно baseline. Добавление attributes не дало дополнительного улучшения по основной метрике, поэтому финальной моделью остаётся V2